# 行为树树状图（缩进示意）

> 本图是 `rmuc_2026.xml` 及所有子树 XML 的结构化缩进视图，用于快速对齐 Groot / BT.CPP 的节点层级。
> 自动生成，请勿手动编辑——如有修改请更新 XML 后重新生成。

## MainSentryTree（主树）

```
rmuc_2026 (主树)
└─ ReactiveSequence
   ├─ SubTree: PerceptionAndBlackboard
   ├─ SubTree: InitOnce
   └─ WhileDoElse (IsMatchStage: game_progress=4, 0~420s)
      ├─ THEN: ReactiveSequence
      │  ├─ SubTree: CommandHub
      │  └─ ReactiveFallback (priority)
      │     ├─ [0]   Sequence { SetBlackboard(active_subtree=RespawnRecovery),   SubTree: RespawnRecovery }
      │     ├─ [0.5] Sequence { SetBlackboard(active_subtree=WeaknessRecovery),  SubTree: WeaknessRecovery }  ← NEW
      │     ├─ [1]   Sequence { SetBlackboard(active_subtree=CriticalSurvival),  SubTree: CriticalSurvival }
      │     ├─ [2]   Sequence { SetBlackboard(active_subtree=BaseDefense),       SubTree: BaseDefense }
      │     ├─ [3]   Sequence { SetBlackboard(active_subtree=EngageCombat),      SubTree: EngageCombat }
      │     ├─ [4]   Sequence { SetBlackboard(active_subtree=SustainAndEconomy), SubTree: SustainAndEconomy }
      │     ├─ [5]   Sequence { SetBlackboard(active_subtree=ObjectivePlanner),  SubTree: ObjectivePlanner }
      │     └─ [6]   Sequence { SetBlackboard(active_subtree=PatrolAndScan),     SubTree: PatrolAndScan }
      └─ ELSE: ReactiveSequence
         ├─ RateController(1Hz) → SendGoal(Home: cfg.home_x/y)
         ├─ RmucRobotControl(stop=T, spin=F, fire=F)
         └─ RateController(2Hz) → SentryCmdMux(posture=3, ...)
```

**优先级说明**（ReactiveFallback 从上到下递减）：

| 优先级 | 子树 | 职责 |
|:---:|:---|:---|
| 0 (最高) | RespawnRecovery | 死亡停车等待 / 虚弱时导航补给区刷卡回血 |
| 0.5 | WeaknessRecovery | **虚弱恢复安全网**：导航最近增益点解除虚弱（NEW） |
| 1 | CriticalSurvival | 危急生存（HP↓ / 热量↑） |
| 2 | BaseDefense | 基地防御（基地受威胁时，含前哨站存活检测） |
| 3 | EngageCombat | 交战（有有效目标且允许战斗，含底盘旋转策略） |
| 4 | SustainAndEconomy | 后勤补给（回血 / 补弹） |
| 5 | ObjectivePlanner | 战略目标占领（~30+ 输入） |
| 6 (最低) | PatrolAndScan | 巡逻扫描 |

> 每个优先级分支均被 `Sequence { SetBlackboard(active_subtree=...), SubTree }` 包裹，
> 使 `active_subtree` 黑板变量始终反映当前正在执行的子树名称。

## 感知 & 初始化子树

### PerceptionAndBlackboard

```
PerceptionAndBlackboard
└─ Sequence
   │
   │  ── 原有 5 个话题订阅 ──
   ├─ RmucSubGameStatus        → {game_status}, {time.now_ms}
   ├─ RmucSubRobotStatus       → {robot_status}
   ├─ RmucSubRFIDStatus        → {rfid.status}
   ├─ RmucSubRobotPosition     → {pose.x/y/yaw}, {is_at_nav_goal}, {pose}
   ├─ SubRadarTracks           → {radar.tracks}
   │
   │  ── P0 新增 7 个裁判系统话题订阅 ──
   ├─ RmucSubSentryDecisionStatus → {sentry_decision_status}   (/sentry_decision_status)
   ├─ RmucSubRobotBuff            → {robot_buff}               (/robot_buff)
   ├─ RmucSubProjectileAllowance  → {projectile_allowance}     (/projectile_allowance)
   ├─ RmucSubFieldStatus          → {field_status}             (/field_status)
   ├─ RmucSubEnemyMark            → {enemy_mark}              (/enemy_mark)
   ├─ RmucSubTeamPositions        → {team_positions}           (/team_positions)
   ├─ RmucSubTeamHP               → {team_hp}                  (/team_hp)
   │
   │  ── 黑板解析 ──
   └─ ParseSentryBlackboard
      inputs (13):
        game_status, robot_status, radar_tracks, pose_x, pose_y, now_ms,
        sentry_decision_status, robot_buff, projectile_allowance,
        field_status, enemy_mark, team_positions, team_hp
      outputs (~70+):
        ── 基础 ──
        {game.remain_s}, {game.elapsed_s},
        {hp.cur}, {hp.max}, {heat.cur},
        {ammo.allow}, {ammo.left},
        {base.hp.cur}, {base.hp.max}, {outpost.alive},
        {state.is_dead}, {state.disengaged}, {state.disengage_cd_s},
        {economy.can_remote_heal}, {economy.can_remote_ammo}, {economy.coins},
        {combat.has_target}, {combat.best_target},
        {threat.base}, {threat.fortress},
        ── P0 新增: 0x020D (哨兵决策状态) ──
        {sentry.can_free_respawn}, {sentry.can_instant_respawn},
        {sentry.instant_respawn_cost}, {sentry.current_posture},
        {sentry.remote_ammo_count}, {sentry.remote_heal_count},
        {sentry.exchanged_ammo_total}, {sentry.can_activate_energy},
        ── P0 新增: 0x0204 (机器人增益) ──
        {buff.heal_rate}, {buff.cool_value}, {buff.defense_pct},
        {buff.vulnerability_pct}, {buff.attack_pct},
        ── P0 新增: 0x0208 (堡垒弹量) ──
        {economy.fortress_ammo},
        ── P0 新增: 0x0101 (场地占领状态) ──
        {field.central_highland}, {field.ladder_highland},
        {field.fortress}, {field.outpost_buff}, {field.base_buff},
        {field.small_energy}, {field.big_energy},
        ── P0 新增: 0x020C (敌方易伤标记) ──
        {enemy.hero_vuln}, {enemy.engi_vuln},
        {enemy.infantry3_vuln}, {enemy.infantry4_vuln},
        {enemy.sentry_vuln},
        ── P0 新增: 0x0003 (队伍建筑HP) ──
        {team.outpost_hp}, {team.base_hp},
        ── P1 新增: 复活/虚弱状态窗口 ──
        {state.respawn_invincible}, {state.respawn_invincible_remain_s},
        {state.power_boosted}, {state.power_boost_remain_s},
      inout:
        {respawn.cum_instant_count}
```

> 📥 共 12 个独立话题订阅 + 1 个黑板解析节点

### InitOnce

```
InitOnce
└─ Sequence
   ├─ InitSentryConfig → 话题名/坐标/阈值 → {cfg.*}
   │    坐标: home, supply_zone, base_buff, outpost_buff,
   │          fortress_ally/enemy, central/ladder_highland,
   │          defend_anchor, patrol_wpt_0/1/2, supply_zone
   │    阈值: arrive_radius, hp_critical/low/safe,
   │          heat_high/critical, ammo_low/target,
   │          base_deficit_for_fortress, enemy_near_base_radius,
   │          objective_hold_ms, combat_fire_burst/pause_ms,
   │          heal_wait_ms, heal_min_ratio, search_timeout_ms
   └─ InitCmdState → {cmd.state}, {cmd.allow_ammo_target}
```

### CommandHub

```
CommandHub
└─ Sequence
   ├─ DecidePosture
   │    inputs: hp_cur/max, heat_cur, heat_high, has_target, base_threat,
   │            is_disengaged, stage_elapsed_time, now_ms, active_subtree,
   │            current_posture, buff_cool_value, buff_defense_pct,
   │            buff_vulnerability_pct, ammo_allow
   │    outputs: {cmd.posture},
   │             {posture.score.attack}, {posture.score.defense}, {posture.score.move}
   │
   ├─ DecideEconomyCmd
   │    inputs: hp_cur/max, ammo_allow, ammo_target, ammo_low,
   │            is_disengaged, can_remote_heal/ammo, team_coins,
   │            stage_remain_time, base_threat,
   │            instant_respawn_cost, cumulative_instant_count,
   │            base_hp_cur/max, fortress_ammo,
   │            remote_heal_count, remote_ammo_count,
   │            allow_ammo_target_in
   │    outputs: {cmd.allow_ammo_target}, {cmd.trig_remote_ammo},
   │             {cmd.trig_remote_hp}, {cmd.enable_big_energy}
   │
   ├─ DecideRespawnCmd
   │    inputs: is_dead, robot_status, team_coins, stage_remain_time,
   │            base_hp_cur/max, base_threat,
   │            can_free_respawn, can_instant_respawn, instant_respawn_cost
   │    inout:  {respawn.cum_instant_count}
   │    outputs: {cmd.confirm_respawn}, {cmd.confirm_instant_respawn}
   │
   └─ RateController(5Hz)
      └─ SentryCmdMux(0x0120)
           posture, confirm_respawn, confirm_instant_respawn,
           allow_ammo_target, trigger_remote_ammo/hp,
           enable_big_energy, cmd_state
```

> ⚠ 相比旧版：`DecidePosture` 新增 score_attack/defense/move 三维评分输出；
> `DecideEconomyCmd` 新增 fortress_ammo、remote_heal/ammo_count 等输入；
> `DecideRespawnCmd` 新增 can_free_respawn、can_instant_respawn 复活可用性输入；
> 末尾 **不再有 KeepRunning**。

## 生存 & 防御子树

### RespawnRecovery（死亡 + 虚弱恢复 一体化）

```
RespawnRecovery
└─ Fallback
   │
   ├─ [A] IfDead_StopAndWait
   │  └─ Sequence
   │     ├─ RmucIsDead
   │     ├─ RmucRobotControl(stop=T, spin=F)
   │     └─ RmucNavControlCmd(cmd_type=3, 原地不动)
   │
   └─ [B] WeaknessRecoveryFlow (ReactiveSequence)
      ├─ IsWeakness ← 门控：不虚弱时 FAILURE → 整棵树退出
      └─ Fallback (RecoveryFlow)
         │
         ├─ IfSupplyCard_Heal
         │  └─ Sequence
         │     ├─ RmucIsSupplyCardDetected
         │     └─ RmucWaitAndHeal (等待回血到 heal_min_ratio)
         │
         └─ GoSupply_ThenSearch
            └─ Sequence
               ├─ RmucNavControlCmd(cmd_type=1, 启动导航)
               ├─ ReactiveFallback (NavUntilArrived)
               │  ├─ RmucIsAtNavGoal
               │  └─ ReactiveSequence
               │     ├─ RateController(5Hz) → SendGoal(GoSupplyToHeal)
               │     └─ KeepRunning
               ├─ RmucRobotControl(stop=T, spin=F)
               ├─ RmucNavControlCmd(cmd_type=3, 刹车)
               ├─ InitSearchTimerIfNeeded
               └─ ReactiveFallback (DetectOrMicroSearch)
                  ├─ RmucIsSupplyCardDetected
                  └─ RmucMicroSearchSupplyCard
```

> **与旧版区别**：外层由 `Sequence { DetectRespawnAndSetRecovery, Fallback }` 改为直接 `Fallback`；
> 复活分支 [B] 改用 `ReactiveSequence` + `IsWeakness` 门控，虚弱恢复后自然退出无需手动清 flag。

状态机：
- 存活 + 不虚弱 → FAILURE（正常退出）
- 死亡(hp=0) → SUCCESS（停车等待复活读条）
- 存活 + 虚弱 → SUCCESS（导航补给区 → 刷卡 → 等回血）

### WeaknessRecovery（虚弱恢复安全网 — NEW）

```
WeaknessRecovery
└─ ReactiveSequence
   ├─ IsWeakness                              ← 门控：不虚弱时立即 FAILURE
   ├─ SelectNearestDispelCard                  → {nav.goal_x/y}
   │    inputs: pose_x/y, supply_zone_x/y, base_buff_x/y, outpost_buff_x/y
   ├─ RateController(1Hz)
   │  └─ SendGoal(DispelWeakness: nav.goal_x/y)
   ├─ RmucRobotControl(stop=F, spin=F, fire=F)
   └─ ReactiveFallback
      ├─ IsAnyDispelCardDetected               ← 到达任意增益点并检测到刷卡
      └─ MoveAround(nearby=5, dis=0.25)        ← 微调搜索
```

> 位于 RespawnRecovery 之后、CriticalSurvival 之前（优先级 0.5）。
> 独立于 RespawnRecovery 的补给区路径，选择最近的增益点（补给区/基地/前哨站）解除虚弱。

### CriticalSurvival

```
CriticalSurvival
└─ ReactiveSequence
   ├─ IsCriticalState (HP < hp_critical 或 heat > heat_critical)
   ├─ RmucRobotControl(stop=F, spin=F, fire=F)
   ├─ SelectSafeRetreatGoal → {nav.goal_x/y}
   │    inputs: pose_x/y, supply_zone_x/y, defend_anchor_x/y
   ├─ RateController(1Hz) → SendGoal(Retreat)
   └─ KeepRunning
```

> ⚠ 与旧版区别：**不再使用 CancelNavGoal**（ReactiveSequence 中每帧取消导航会冲突）；
> `SelectSafeRetreatGoal` 使用 `supply_zone_x/y`（非 base_buff）和 `defend_anchor_x/y`。

### BaseDefense

```
BaseDefense
└─ ReactiveSequence
   ├─ IsBaseThreatened
   │    inputs: base_threat, base_hp_cur/max, enemy_near_base_radius,
   │            outpost_alive  ← 前哨站存活 → 基地无敌 → 降低威胁敏感度
   ├─ RmucRobotControl(stop=F, spin=F, fire=T)
   ├─ RateController(1Hz) → SendGoal(DefendAnchor: cfg.defend_anchor_x/y)
   └─ SubTree: CombatLoop
```

## 战斗子树

### EngageCombat

```
EngageCombat
└─ ReactiveSequence
   ├─ HasValidTarget (has_target, best_target)
   ├─ IsCombatAllowed (robot_status, ammo, heat, hp)
   │
   │  ── nav.goal 覆写（锁定当前位置，使 dist≈0 允许旋转） ──
   ├─ SetBlackboard(nav.goal_x ← pose.x)
   ├─ SetBlackboard(nav.goal_y ← pose.y)
   │
   │  ── 底盘旋转策略 ──
   ├─ ReactiveFallback
   │  ├─ Sequence
   │  │  ├─ ShouldChassisSpin
   │  │  │    inputs: pose_x/y, goal_x/y, arrive_radius,
   │  │  │            current_posture, is_power_boosted
   │  │  └─ RmucRobotControl(stop=T, spin=T, fire=T)
   │  └─ RmucRobotControl(stop=T, spin=F, fire=T)  ← 不满足旋转条件时
   │
   └─ SubTree: CombatLoop
```

> ⚠ 与旧版区别：新增 `SetBlackboard` 覆写 nav.goal 为当前位置；
> 新增 `ShouldChassisSpin` 条件节点 + `ReactiveFallback` 旋转策略分支。

### CombatLoop

```
CombatLoop
└─ ReactiveSequence
   ├─ SelectBestTarget → {combat.best_target}
   │    inputs: radar_tracks, pose_x/y, base_x/y,
   │            enemy_hero_vuln, enemy_engi_vuln,
   │            enemy_infantry3/4_vuln, enemy_sentry_vuln
   ├─ AimAtTarget (target=combat.best_target)
   ├─ ReactiveFallback
   │  ├─ IsFireWindowOk
   │  │    inputs: heat_cur, heat_high, ammo_allow, robot_status,
   │  │            current_posture, buff_cool_value,
   │  │            buff_vulnerability_pct, ammo_conserve=30
   │  └─ RmucRobotControl(fire=F)
   ├─ FireBurst(burst_ms, pause_ms)
   └─ KeepRunning
```

> ⚠ 与旧版区别：`SelectBestTarget` 新增敌方易伤 (vulnerability) 输入 (0x020C)；
> `IsFireWindowOk` 新增 current_posture、buff_cool_value、buff_vulnerability_pct、ammo_conserve。

## 后勤子树

### SustainAndEconomy

```
SustainAndEconomy
└─ ReactiveFallback
   ├─ SubTree: HealPlan
   └─ SubTree: AmmoPlan
```

### HealPlan

```
HealPlan
└─ ReactiveSequence
   ├─ RmucIsHPBelow(hp_threshold={cfg.hp_low})
   └─ ReactiveFallback
      ├─ (已在补给区) → ReactiveSequence
      │  ├─ IsZoneCardDetected(zone="SUPPLY")
      │  └─ HoldAndHeal (驻留至 hp ≥ hp_safe)
      └─ (前往补给区) → ReactiveSequence
         ├─ RateController(1Hz) → SendGoal(GoSupply: cfg.supply_zone_x/y)
         ├─ RmucRobotControl(stop=F, spin=F, fire=F)
         └─ KeepRunning
```

### AmmoPlan

```
AmmoPlan
└─ ReactiveSequence
   ├─ IsAmmoBelow(ammo_low={cfg.ammo_low})
   └─ ReactiveFallback
      ├─ (已在补给区) → ReactiveSequence
      │  ├─ IsZoneCardDetected(zone="SUPPLY")
      │  └─ HoldForSupplyAmmoTick (等待弹量达标)
      └─ (前往最近补给站) → ReactiveSequence
         ├─ SelectNearestResupplyStation → {nav.goal_x/y}
         │    inputs: pose_x/y, supply_zone_x/y, base_buff_x/y, outpost_buff_x/y
         ├─ RateController(1Hz) → SendGoal(GoResupplyStation)
         ├─ RmucRobotControl(stop=F, spin=F, fire=F)
         └─ KeepRunning
```

## 目标 & 巡逻子树

### ObjectivePlanner

```
ObjectivePlanner
└─ ReactiveSequence
   ├─ SelectObjective → {nav.objective}, {nav.goal_x/y}
   │    inputs (~30+):
   │      pose_x/y, stage_elapsed/remain_time,
   │      hp_cur/max, ammo_allow, ammo_target,
   │      base_hp_cur/max, base_deficit_for_fortress,
   │      outpost_alive, base_threat,
   │      field_central_highland, field_ladder_highland,
   │      field_fortress, field_outpost_buff, field_base_buff,
   │      fortress_ammo,
   │      central_highland_x/y, ladder_highland_x/y,
   │      base_buff_x/y, outpost_buff_x/y,
   │      fortress_ally_x/y, fortress_enemy_x/y,
   │      supply_zone_x/y, defend_anchor_x/y
   ├─ RateController(1Hz) → SendGoal({nav.objective})
   ├─ RmucRobotControl(stop=F, spin=F, fire=F)
   ├─ ReactiveFallback
   │  ├─ IsAtGoal(arrive_radius)
   │  └─ KeepRunning
   └─ HoldObjective(hold_ms, base_threat, has_target)
```

> ⚠ 与旧版区别：`SelectObjective` 新增 field_* 占领状态 (0x0101)、fortress_ammo、
> 多组候选目标坐标（central_highland, ladder_highland, base_buff, outpost_buff,
> fortress_ally/enemy, supply_zone, defend_anchor）共 ~30+ 个输入端口。

### PatrolAndScan

```
PatrolAndScan
└─ ReactiveSequence
   ├─ RmucRobotControl(stop=F, spin=F, fire=F)
   ├─ WaypointPatrol → {nav.goal_x/y}
   │    inputs: pose_x/y, wpt0/1/2_x/y
   ├─ RateController(1Hz) → SendGoal(Patrol)
   └─ KeepRunning
```

> ⚠ 与旧版区别：外层由 `Sequence` 改为 `ReactiveSequence`。